# DataCo | Modeling Pipeline

**Modelo de Clasificación del Riesgo de Entrega Tardía**

## 1. Carga y preparación inicial de datos

En esta sección se importan las bibliotecas necesarias y se carga el conjunto de datos DataCo Smart Supply Chain.

El objetivo es establecer la base del flujo de preparación de datos para el modelo de clasificación, cuya variable objetivo es `Late_delivery_risk`.


In [2]:

import pandas as pd
import numpy as np
from pathlib import Path

In [3]:

# Localizar el conjunto de datos
rutas_dataset = [
    Path("Dataset/DataCoSupplyChainDataset.csv"),
    Path("DataCoSupplyChainDataset.csv")
]

ruta_dataset = next(
    (ruta for ruta in rutas_dataset if ruta.exists()),
    None
)

if ruta_dataset is None:
    raise FileNotFoundError(
        "No se encontró DataCoSupplyChainDataset.csv. "
        "Colóquelo en la carpeta Dataset/ o junto al notebook."
    )

# Cargar el dataset
df = pd.read_csv(
    ruta_dataset,
    encoding="latin1",
    sep=";",
    low_memory=False
)

df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Name,Customer City,Customer Country,...,Order State,Order Status,Order Zipcode,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,Sporting Goods,Caguas,Puerto Rico,...,Java Occidental,COMPLETE,NaN,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,Sporting Goods,Caguas,Puerto Rico,...,Rajastï¿,PENDING,NaN,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,Sporting Goods,San Jose,EE. UU.,...,Rajastï¿,CLOSED,NaN,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,Sporting Goods,Los Angeles,EE. UU.,...,Queensland,COMPLETE,NaN,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,Sporting Goods,Caguas,Puerto Rico,...,Queensland,PENDING_PAYMENT,NaN,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [4]:

print(f"Registros: {df.shape[0]:,}")
print(f"Variables: {df.shape[1]}")

Registros: 180,519
Variables: 44


## 2. División estratificada de entrenamiento y prueba

Se divide el conjunto de datos en dos subconjuntos: 80 % para entrenamiento y 20 % para prueba.

Se utiliza un muestreo estratificado basado en la variable `Shipping Mode`, con el propósito de conservar la distribución de las modalidades de envío en ambos conjuntos.

Se establece `random_state=42` para garantizar la reproducibilidad de la división.


In [5]:

from sklearn.model_selection import train_test_split

# Definir variable objetivo
variable_objetivo = "Late_delivery_risk"

# Separar variables predictoras y variable objetivo
X = df.drop(columns=[variable_objetivo])
y = df[variable_objetivo]

# División estratificada
X_train_estratificado, X_test_estratificado, y_train_estratificado, y_test_estratificado = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=X["Shipping Mode"]
)

print(f"Entrenamiento: {len(X_train_estratificado):,} registros")
print(f"Prueba: {len(X_test_estratificado):,} registros")
print(f"Variable objetivo: {variable_objetivo}")

Entrenamiento: 144,415 registros
Prueba: 36,104 registros
Variable objetivo: Late_delivery_risk


## 3. Preparación del conjunto de entrenamiento

Se preparan los conjuntos de datos obtenidos mediante la división estratificada, manteniendo separadas las variables predictoras y la variable objetivo `Late_delivery_risk`.

El conjunto de entrenamiento se utilizará para ajustar los transformadores de preprocesamiento, mientras que el conjunto de prueba permanecerá reservado para evaluar posteriormente el modelo.

Las transformaciones se aplicarán únicamente a las características seleccionadas, excluyendo aquellas que puedan producir fuga de información.


In [6]:

# Preparar los conjuntos de entrenamiento y prueba
df_train_limpieza = X_train_estratificado.copy()
df_test_limpieza = X_test_estratificado.copy()

# Conservar las etiquetas por separado
df_train_labels = y_train_estratificado.copy()
df_test_labels = y_test_estratificado.copy()

print("Conjunto de entrenamiento:", df_train_limpieza.shape)
print("Conjunto de prueba:", df_test_limpieza.shape)
print("Etiquetas de entrenamiento:", df_train_labels.shape)
print("Etiquetas de prueba:", df_test_labels.shape)

Conjunto de entrenamiento: (144415, 43)
Conjunto de prueba: (36104, 43)
Etiquetas de entrenamiento: (144415,)
Etiquetas de prueba: (36104,)


## 4. Implementación de transformadores personalizados

Se definen transformadores personalizados para incorporar nuevas características al proceso de preparación de datos de DataCo.

Se implementan dos mecanismos de transformación:

- **ClusterSimilarity:** utiliza K-Means y funciones de base radial (RBF) para calcular la similitud de cada registro con los centroides de los clústeres identificados.
- **Transformación de ratios:** permite generar características derivadas mediante la relación entre dos atributos numéricos, como descuentos, beneficios e ingresos por unidad.

Estos transformadores se integrarán posteriormente en los pipelines de preprocesamiento para automatizar la ingeniería de características.


In [7]:

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans

class ClusterSimilarity(BaseEstimator, TransformerMixin):

    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(
            n_clusters=self.n_clusters,
            n_init=10,
            random_state=self.random_state
        )

        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(
            X,
            self.kmeans_.cluster_centers_,
            gamma=self.gamma
        )

    def get_feature_names_out(self, names=None):
        return [
            f"Cluster {i} similarity"
            for i in range(self.n_clusters)
        ]
        
        
# Calcular ratio entre dos columnas
def column_ratio(X):
    numerador = X[:, [0]].astype(float)
    denominador = X[:, [1]].astype(float)

    return np.divide(
        numerador,
        denominador,
        out=np.full_like(numerador, np.nan),
        where=denominador != 0
    )

# Nombre de la columna generada
def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]

print("Transformadores personalizados definidos correctamente.")


Transformadores personalizados definidos correctamente.


## 5. Definición de pipelines de preprocesamiento

Se implementan pipelines de Scikit-learn para automatizar las transformaciones necesarias en la preparación de los datos de DataCo.

Los procesos contemplan:

- **Atributos numéricos:** imputación de valores faltantes mediante la mediana y estandarización con StandardScaler.
- **Ratios:** generación de características derivadas mediante divisiones entre atributos, seguida de imputación y estandarización.
- **Transformaciones logarítmicas:** aplicación de log1p para transformar atributos no negativos y posteriormente estandarizarlos.
- **Atributos categóricos:** imputación mediante la categoría más frecuente y codificación con OneHotEncoder.
- **Similitud geográfica:** estandarización de coordenadas y generación de características mediante ClusterSimilarity.

Estos pipelines serán integrados mediante ColumnTransformer para aplicar cada transformación a los atributos correspondientes.


In [8]:

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    FunctionTransformer,
    StandardScaler,
    OneHotEncoder
)

# Pipeline para atributos numéricos
num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

# Pipeline para generar ratios
def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(
            column_ratio,
            feature_names_out=ratio_name
        ),
        SimpleImputer(strategy="median"),
        StandardScaler()
    )

# Pipeline para transformaciones logarítmicas
log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(
        np.log1p,
        feature_names_out="one-to-one"
    ),
    StandardScaler()
)

# Pipeline para ratios con transformación logarítmica
def log_ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(
            column_ratio,
            feature_names_out=ratio_name
        ),
        SimpleImputer(strategy="median"),
        FunctionTransformer(
            np.log1p,
            feature_names_out="one-to-one"
        ),
        StandardScaler()
    )

# Pipeline para atributos categóricos
cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
)

# Pipeline para similitud geográfica
geo_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    ClusterSimilarity(
        n_clusters=10,
        gamma=1.0,
        random_state=42
    )
)

print("Pipelines de preprocesamiento definidos correctamente.")


Pipelines de preprocesamiento definidos correctamente.


## 6. Integración de transformaciones mediante ColumnTransformer

Se utiliza `ColumnTransformer` de Scikit-learn para integrar los pipelines de preprocesamiento y asignar las transformaciones correspondientes a cada grupo de atributos del conjunto de datos DataCo.

El proceso contempla el tratamiento de variables numéricas, la generación de ratios, las transformaciones logarítmicas, la codificación de atributos categóricos y la generación de características de similitud geográfica mediante K-Means.

Se conservan las variables originales seleccionadas y se incorporan las características derivadas. Los atributos no incluidos en las transformaciones se excluyen mediante `remainder="drop"`.

Esta configuración permite centralizar el preprocesamiento y aplicar las transformaciones de manera consistente durante las siguientes etapas del modelado.


In [9]:

from sklearn.compose import ColumnTransformer

# Variables numéricas originales
columnas_numericas = [
    "Days for shipment (scheduled)",
    "Order Item Quantity",
    "Sales",
    "Order Item Total",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Product Price",
    "Benefit per order",
    "Order Item Profit Ratio",
    "Latitude",
    "Longitude"
]

# Variables categóricas
columnas_categoricas = [
    "Type",
    "Customer Country",
    "Customer Segment",
    "Market",
    "Department Name",
    "Shipping Mode"
]

# Variables con transformación logarítmica
columnas_log = [
    "Sales",
    "Order Item Total",
    "Order Item Discount",
    "Product Price"
]

# Integrar los pipelines
preprocessing = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, columnas_numericas),

        ("discount_ratio", ratio_pipeline(),
         ["Order Item Discount", "Order Item Quantity"]),

        ("profit_ratio", ratio_pipeline(),
         ["Benefit per order", "Order Item Quantity"]),

        ("revenue_ratio", ratio_pipeline(),
         ["Order Item Total", "Order Item Quantity"]),

        ("log", log_pipeline, columnas_log),

        ("log_discount_ratio", log_ratio_pipeline(),
         ["Order Item Discount", "Order Item Quantity"]),

        ("log_revenue_ratio", log_ratio_pipeline(),
         ["Order Item Total", "Order Item Quantity"]),

        ("geo", geo_pipeline, ["Latitude", "Longitude"]),

        ("cat", cat_pipeline, columnas_categoricas)
    ],
    remainder="drop"
)

print("ColumnTransformer configurado correctamente.")
print("\nTransformaciones definidas:")

for nombre, transformador, columnas in preprocessing.transformers:
    print(f"- {nombre}: {columnas}")


ColumnTransformer configurado correctamente.

Transformaciones definidas:
- num: ['Days for shipment (scheduled)', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Item Discount', 'Order Item Discount Rate', 'Product Price', 'Benefit per order', 'Order Item Profit Ratio', 'Latitude', 'Longitude']
- discount_ratio: ['Order Item Discount', 'Order Item Quantity']
- profit_ratio: ['Benefit per order', 'Order Item Quantity']
- revenue_ratio: ['Order Item Total', 'Order Item Quantity']
- log: ['Sales', 'Order Item Total', 'Order Item Discount', 'Product Price']
- log_discount_ratio: ['Order Item Discount', 'Order Item Quantity']
- log_revenue_ratio: ['Order Item Total', 'Order Item Quantity']
- geo: ['Latitude', 'Longitude']
- cat: ['Type', 'Customer Country', 'Customer Segment', 'Market', 'Department Name', 'Shipping Mode']


## 7. Ejecución del preprocesamiento

Se ejecuta el flujo de preprocesamiento mediante `fit_transform()` sobre el conjunto de entrenamiento de DataCo.

El proceso aplica automáticamente las transformaciones configuradas en `ColumnTransformer`, incluyendo imputación, estandarización, generación de ratios, transformaciones logarítmicas, codificación categórica y similitud geográfica.

La matriz resultante se convierte en un DataFrame de pandas utilizando los nombres de las características generadas, con el propósito de facilitar su interpretación.

El resultado constituye el conjunto de características preparado para el entrenamiento posterior del modelo de clasificación de riesgo de entrega tardía.


In [10]:

# Ejecutar el preprocesamiento sobre los datos de entrenamiento
X_train_preparado = preprocessing.fit_transform(
    df_train_limpieza
)

# Obtener los nombres de las características generadas
nombres_caracteristicas = preprocessing.get_feature_names_out()

# Convertir la matriz procesada a DataFrame
df_train_procesado = pd.DataFrame(
    X_train_preparado,
    columns=nombres_caracteristicas,
    index=df_train_limpieza.index
)

# Mostrar información del resultado
print("Preprocesamiento completado correctamente.")
print("Tipo de matriz:", type(X_train_preparado))
print("Dimensiones originales:", df_train_limpieza.shape)
print("Dimensiones procesadas:", df_train_procesado.shape)
print("Cantidad de características:", len(nombres_caracteristicas))

# Visualizar las primeras filas
display(df_train_procesado.head(3))


Preprocesamiento completado correctamente.
Tipo de matriz: <class 'numpy.ndarray'>
Dimensiones originales: (144415, 43)
Dimensiones procesadas: (144415, 59)
Cantidad de características: 59


,num__Days for shipment (scheduled),num__Order Item Quantity,num__Sales,num__Order Item Total,num__Order Item Discount,num__Order Item Discount Rate,num__Product Price,num__Benefit per order,num__Order Item Profit Ratio,num__Latitude,...,cat__Department Name_Footwear,cat__Department Name_Golf,cat__Department Name_Health and Beauty,cat__Department Name_Outdoors,cat__Department Name_Pet Shop,cat__Department Name_Technology,cat__Shipping Mode_First Class,cat__Shipping Mode_Same Day,cat__Shipping Mode_Second Class,cat__Shipping Mode_Standard Class
32071,-0.677969,0.599535,-1.201388,-1.227771,-0.533503,1.399276,-0.904599,-0.767414,-3.676678,-1.170107,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
83230,0.777157,-0.775773,-0.557277,-0.453094,-0.885832,-1.301431,-0.078932,0.397092,0.790646,0.395909,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
164640,0.777157,1.287188,-0.026882,-0.074487,0.245835,0.404278,-0.653237,-0.085721,-0.085719,1.082728,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## 8. Entrenamiento y evaluación de clasificadores

Se entrenan algoritmos de clasificación integrando cada uno en un pipeline junto con `preprocessing`. Los modelos se evalúan mediante validación cruzada (`cross_val_score`) con 3 particiones y la métrica `accuracy`, utilizando los hiperparámetros por defecto. El conjunto de prueba permanece reservado.

In [12]:
### 8.1 Clasificador SGD (descenso de gradiente estocástico)
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import cross_val_score

# Pipeline: primero preprocesa y después clasifica
sgd_clf = make_pipeline(
    preprocessing,
    SGDClassifier(random_state=42)
)

# Entrenamiento con validación cruzada
sgd_scores = cross_val_score(
    sgd_clf,
    df_train_limpieza,
    df_train_labels,
    cv=3,
    scoring="accuracy"
)

print("Exactitud por partición:", sgd_scores.round(4))
print(f"Exactitud promedio: {sgd_scores.mean():.4f}")

Exactitud por partición: [0.6981 0.6919 0.6955]
Exactitud promedio: 0.6952


In [13]:
### 8.2 Random Forest (bosque aleatorio)
from sklearn.ensemble import RandomForestClassifier

forest_clf = make_pipeline(
    preprocessing,
    RandomForestClassifier(random_state=42)
)

forest_scores = cross_val_score(
    forest_clf,
    df_train_limpieza,
    df_train_labels,
    cv=3,
    scoring="accuracy"
)

print("Exactitud por partición:", forest_scores.round(4))
print(f"Exactitud promedio: {forest_scores.mean():.4f}")

Exactitud por partición: [0.7162 0.7085 0.7119]
Exactitud promedio: 0.7122


In [14]:
### 8.3 Clasificador de referencia (DummyClassifier)
from sklearn.dummy import DummyClassifier

# Clasificador "tonto": siempre predice la clase más frecuente
dummy_clf = DummyClassifier(strategy="most_frequent")

dummy_scores = cross_val_score(
    dummy_clf,
    df_train_limpieza,
    df_train_labels,
    cv=3,
    scoring="accuracy"
)

print(f"Exactitud promedio del DummyClassifier: {dummy_scores.mean():.4f}")

Exactitud promedio del DummyClassifier: 0.5482


In [15]:
### 8.4 Comparación de resultados
comparacion_modelos = pd.DataFrame({
    "Modelo": ["DummyClassifier", "SGDClassifier", "RandomForestClassifier"],
    "Exactitud promedio": [dummy_scores.mean(), sgd_scores.mean(), forest_scores.mean()],
    "Desviación estándar": [dummy_scores.std(), sgd_scores.std(), forest_scores.std()]
})

display(comparacion_modelos.round(4))

,Modelo,Exactitud promedio,Desviación estándar
0,DummyClassifier,0.5482,0.0000
1,SGDClassifier,0.6952,0.0026
2,RandomForestClassifier,0.7122,0.0032


### 8.5 Interpretación de los resultados

El DummyClassifier, que siempre predice "entrega tardía", obtuvo una exactitud de 54.82 %. Este valor representa el mínimo que un modelo útil debe superar.

Ambos modelos superan claramente esta referencia: el SGDClassifier obtuvo 69.52 % y el RandomForestClassifier 71.22 %, siendo este último el de mejor desempeño. Las desviaciones estándar son muy bajas, por lo que ambos modelos son estables entre las particiones.

La exactitud no indica qué tipo de errores comete el modelo. En las siguientes etapas se evaluarán la matriz de confusión, la precisión, el recall y el F1-score.